# EDER x ZOTTER — Colab Starter (Gemini API + LCA Test)
Secrets: Key-Icon → `GEMINI_API_KEY`, Notebook access ON. Never paste keys in cells.

In [ ]:
!pip -q install google-genai pandas matplotlib

In [ ]:
from google.colab import userdata
import os
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')  # key from Colab Secrets, never in cell

In [ ]:
from google import genai
client = genai.Client()  # reads GEMINI_API_KEY
MODEL = 'gemini-3.5-flash'
r = client.models.generate_content(model=MODEL, contents='Summarize ISO 20887 DfD principles in 5 bullets for a slide.')
print(r.text)

## Test 2 — LCA Tabelle Tallinn: Masse × CO2-Faktor

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Eingabe: Bauteil, Masse [kg], GWP-Faktor [kgCO2e/kg] — Platzhalterwerte, durch EPD-Werte ersetzen
df = pd.DataFrame({
    'element': ['CLT wall','Concrete slab','Steel beam','Glulam beam'],
    'mass_kg': [12000.0, 45000.0, 3000.0, 6500.0],
    'gwp_kgco2e_per_kg': [-0.60, 0.11, 1.50, -0.55],  # negativ = biogener Speicher (A1-A3, Platzhalter)
})
df['gwp_kgco2e'] = df['mass_kg'] * df['gwp_kgco2e_per_kg']
df.loc['TOTAL'] = ['TOTAL', df['mass_kg'].sum(), float('nan'), df['gwp_kgco2e'].sum()]
df

In [ ]:
ax = df.iloc[:-1].plot.bar(x='element', y='gwp_kgco2e', legend=False)
ax.axhline(0, color='k', lw=0.8)
ax.set_ylabel('GWP A1–A3 [kgCO2e]')
ax.set_title('Tallinn LCA — Masse × Faktor (Platzhalterdaten)')
plt.tight_layout(); plt.show()

## Structured Output (JSON-Schema) — für Slide-Pipeline

In [ ]:
from pydantic import BaseModel
class Claim(BaseModel):
    slide: int
    text: str
    absolute: bool  # '0%', 'infinite', 'no glue' ...
r = client.models.generate_content(
    model=MODEL,
    contents='Slide 3: Our CLT wall has 0% waste and infinite reuse. Slide 4: Screws only, no glue.',
    config={'response_mime_type':'application/json','response_schema':list[Claim]},
)
print(r.text)